# Q1 — Reproducible Data Pipeline

Builds the unified-schema feature store described in `SPEC.md` from the raw
EB-NeRD demo, EB-NeRD small, EB-NeRD large, MIND-small, and MIND-large
files. Each step below unifies one raw table into the shared schema, then
asserts the invariants that make it safe to build on top of (Q2-Q4) and free
of the leakage patterns called out in Q9.

Processed as two fully separate families, EB-NeRD then MIND -- each one
built, split, leakage-checked, written, and freed completely before the
other starts -- rather than interleaved table-by-table across all five
datasets. This is a memory constraint, not a stylistic choice: at
`ebnerd_large`/`mind_large` scale, holding both families' tables in memory
at once OOM-killed the kernel on a 16GB machine (see the write cells'
markdown for the full story).

**`BUILD_LARGE_ONLY`** (set in the setup cell below): when `True`,
`ebnerd`/`ebnerd_small`/`mind` are skipped entirely during execution --
their `data/processed/` outputs already exist and are correct from earlier
runs, so re-building and re-writing them every time this notebook is
iterated on (while debugging `ebnerd_large`/`mind_large`'s memory issues)
is both wasteful and a needless risk to already-good results.
`write_feature_store` is simply never *called* for those three datasets in
this mode, so their existing `data/processed/{name}/` directories are never
touched. Every cell below still contains the demo/small/mind-regular code
path, guarded by `if not BUILD_LARGE_ONLY:` -- flip the flag back to
`False` for a full five-dataset rebuild.

Run top-to-bottom (or via `python build_pipeline.py`, which executes this
notebook end-to-end) to rebuild `data/processed/` from raw files.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import gc
import json
import shutil

import numpy as np
import pandas as pd
import polars as pl


def find_repo_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"could not locate {marker} above {Path.cwd()}")


ROOT = find_repo_root()
EBNERD_DEMO = ROOT / "ebnerd_demo"
EBNERD_SMALL = ROOT / "ebnerd_small"
EBNERD_LARGE = ROOT / "ebnerd_large"
MIND_TRAIN = ROOT / "MINDsmall_train" / "MINDsmall_train"
MIND_DEV = ROOT / "MINDsmall_dev" / "MINDsmall_dev"
MIND_LARGE_TRAIN = ROOT / "MINDlarge_train" / "MINDlarge_train"
MIND_LARGE_DEV = ROOT / "MINDlarge_dev" / "MINDlarge_dev"
DATA_OUT = ROOT / "data" / "processed"

# When True, skip building/writing ebnerd/ebnerd_small/mind entirely -- their
# data/processed/ outputs already exist and are correct from earlier runs.
# write_feature_store is never called for those three in this mode, so their
# directories are never touched. See the notebook's intro markdown for why.
BUILD_LARGE_ONLY = True

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

# Plain-text progress log, external to the notebook's own saved output --
# nbconvert only writes build_pipeline.ipynb back to disk once, at the very
# end of a successful run, so it can't be tailed mid-run. This file is
# appended to (and flushed) after every major step below instead, so
# progress can be watched live regardless of notebook/nbconvert internals
# (e.g. `Get-Content build_progress.log -Wait -Tail 20` in PowerShell, or
# `tail -f build_progress.log` in bash). Truncated at the start of each run.
PROGRESS_LOG = ROOT / "build_progress.log"
PROGRESS_LOG.write_text("", encoding="utf-8")


def log_progress(message: str) -> None:
    line = f"[{datetime.now(timezone.utc).isoformat(timespec='seconds')}] {message}"
    print(line)
    with open(PROGRESS_LOG, "a", encoding="utf-8") as f:
        f.write(line + "\n")


log_progress(f"build_pipeline started (BUILD_LARGE_ONLY={BUILD_LARGE_ONLY})")
ROOT

[2026-08-21T06:44:26+00:00] build_pipeline started (BUILD_LARGE_ONLY=True)


WindowsPath('C:/Users/HP/cs4406m26-assignment1c1')

## EB-NeRD -> unified `articles`

Maps EB-NeRD's `subtitle` to the shared `abstract` column (its closest
analogue to MIND's abstract) and keeps `body`, which MIND never has.

Raw loading and the prefix-namespacing transforms below use `polars`, not
`pandas.apply()`/`.map()` -- at `ebnerd_large`'s scale (~24.6M behavior rows,
125,541 articles), per-row Python-level list/string operations are the
actual bottleneck (Python object overhead per list element, not I/O), and
`polars`' expression API (`list.eval`, `pl.element()`) does the same
transforms as vectorized, multi-threaded Rust operations instead. Each
`build_*` function still returns a plain `pandas.DataFrame` (`.to_pandas()`
at the end), so every downstream cell (temporal split, leakage checks,
persistence, tests) is unchanged and still operates on pandas.

In [2]:
def build_ebnerd_articles(raw: pl.DataFrame, prefix: str) -> pd.DataFrame:
    out = raw.select(
        (pl.lit(prefix) + pl.col("article_id").cast(pl.Utf8)).alias("article_id"),
        pl.lit(prefix.rstrip("_")).alias("dataset"),
        pl.col("title"),
        pl.col("subtitle").alias("abstract"),
        pl.col("body"),
        pl.col("category_str").alias("category"),
        pl.col("subcategory").list.first().cast(pl.Utf8).alias("subcategory"),
        pl.col("published_time"),
    )
    return out.to_pandas()


if not BUILD_LARGE_ONLY:
    ebnerd_articles_raw = pl.read_parquet(EBNERD_DEMO / "articles.parquet")
    ebnerd_articles = build_ebnerd_articles(ebnerd_articles_raw, "ebnerd_")
    log_progress(f"EB-NeRD demo articles built: {ebnerd_articles.shape}")

    ebnerd_small_articles_raw = pl.read_parquet(EBNERD_SMALL / "articles.parquet")
    ebnerd_small_articles = build_ebnerd_articles(ebnerd_small_articles_raw, "ebnerd_small_")
    log_progress(f"EB-NeRD small articles built: {ebnerd_small_articles.shape}")

ebnerd_large_articles_raw = pl.read_parquet(EBNERD_LARGE / "articles.parquet")
ebnerd_large_articles = build_ebnerd_articles(ebnerd_large_articles_raw, "ebnerd_large_")
log_progress(f"EB-NeRD large articles built: {ebnerd_large_articles.shape}")

if not BUILD_LARGE_ONLY:
    print("ebnerd shape:", ebnerd_articles.shape)
    print("ebnerd_small shape:", ebnerd_small_articles.shape)
print("ebnerd_large shape:", ebnerd_large_articles.shape)
ebnerd_large_articles.head(5)

[2026-08-21T06:44:27+00:00] EB-NeRD large articles built: (125541, 8)
ebnerd_large shape: (125541, 8)


,article_id,dataset,title,abstract,body,category,subcategory,published_time
0,ebnerd_large_3000022,ebnerd_large,Hanks beskyldt for mishandling,Tom Hanks har angiveligt mishandlet sin afdøde ekskone. Ny bog om skuespille...,"Tom Hanks skulle angiveligt have mishandlet sin ekskone, Samantha Lewes.\nDe...",underholdning,432,2006-09-20 09:24:18
1,ebnerd_large_3000063,ebnerd_large,Bostrups aske spredt i Furesøen,Studieværten blev mindet med glad festlighed,Strålende sensommersol. Jazzede toner. Glas med champagne og hvidvin. Glade ...,nyheder,133,2006-09-24 07:45:30
2,ebnerd_large_3000613,ebnerd_large,Jesper Olsen ramt af hjerneblødning,Den tidligere danske landsholdsspiller i fodbold Jesper Olsen ligger på hosp...,"Jesper Olsen, der er noteret for 43 kampe på det danske fodboldlandshold, li...",sport,196,2006-05-09 11:29:00
3,ebnerd_large_3000700,ebnerd_large,Madonna topløs med heste,47-årige Madonna poserer både topløs og sammen med heste i den kommende udga...,Skal du have stillet Madonna-sulten inden koncerten i Horsens den 24. august...,underholdning,432,2006-05-04 11:03:12
4,ebnerd_large_3000840,ebnerd_large,Otto Brandenburg er død,"Sangeren og skuespilleren Otto Brandenburg er død, 72 år, efter længere tids...","'Og lidt for Susanne, Birgitte og Hanne... ' 'To lys på et bord, tre forløse...",nyheder,133,2007-03-01 18:34:00


In [3]:
def test_ebnerd_articles_unified_schema():
    expected_cols = {
        "article_id", "dataset", "title", "abstract", "body",
        "category", "subcategory", "published_time",
    }
    cases = [(ebnerd_large_articles, ebnerd_large_articles_raw, "ebnerd_large_", "ebnerd_large")]
    if not BUILD_LARGE_ONLY:
        cases = [
            (ebnerd_articles, ebnerd_articles_raw, "ebnerd_", "ebnerd"),
            (ebnerd_small_articles, ebnerd_small_articles_raw, "ebnerd_small_", "ebnerd_small"),
        ] + cases
    for articles, raw, prefix, dataset_name in cases:
        assert expected_cols == set(articles.columns)
        assert len(articles) == len(raw)
        assert articles["article_id"].is_unique
        assert (articles["dataset"] == dataset_name).all()
        assert articles["article_id"].str.startswith(prefix).all()


test_ebnerd_articles_unified_schema()

# free the raw polars frames -- not referenced anywhere past this point
# (memory matters at ebnerd_large's scale: peak RSS OOM-killed the kernel
# during the first attempt at this pipeline, on this machine's 16GB RAM)
if BUILD_LARGE_ONLY:
    del ebnerd_large_articles_raw
else:
    del ebnerd_articles_raw, ebnerd_small_articles_raw, ebnerd_large_articles_raw
gc.collect()

print("ok: EB-NeRD unified articles schema checks passed")

ok: EB-NeRD unified articles schema checks passed


## EB-NeRD -> unified `behaviors`

Combines the provider's `train/` and `validation/` impression logs into one
table - our own temporal split (below) decides `train`/`val`/`test`, not the
provider's file layout.

In [4]:
def build_ebnerd_behaviors(raw: pl.DataFrame, prefix: str) -> pd.DataFrame:
    out = raw.select(
        (pl.lit(prefix) + pl.col("impression_id").cast(pl.Utf8)).alias("impression_id"),
        pl.lit(prefix.rstrip("_")).alias("dataset"),
        (pl.lit(prefix) + pl.col("user_id").cast(pl.Utf8)).alias("user_id"),
        pl.col("impression_time"),
        pl.col("article_ids_inview").list.eval(pl.lit(prefix) + pl.element().cast(pl.Utf8)).alias("article_ids_inview"),
        pl.col("article_ids_clicked").list.eval(pl.lit(prefix) + pl.element().cast(pl.Utf8)).alias("article_ids_clicked"),
        (pl.lit(prefix) + pl.col("session_id").cast(pl.Utf8)).alias("session_id"),
    )
    return out.to_pandas()


if not BUILD_LARGE_ONLY:
    ebnerd_behaviors_raw = pl.concat([
        pl.read_parquet(EBNERD_DEMO / "train" / "behaviors.parquet"),
        pl.read_parquet(EBNERD_DEMO / "validation" / "behaviors.parquet"),
    ])
    ebnerd_behaviors = build_ebnerd_behaviors(ebnerd_behaviors_raw, "ebnerd_")
    log_progress(f"EB-NeRD demo behaviors built: {ebnerd_behaviors.shape}")

    ebnerd_small_behaviors_raw = pl.concat([
        pl.read_parquet(EBNERD_SMALL / "train" / "behaviors.parquet"),
        pl.read_parquet(EBNERD_SMALL / "validation" / "behaviors.parquet"),
    ])
    ebnerd_small_behaviors = build_ebnerd_behaviors(ebnerd_small_behaviors_raw, "ebnerd_small_")
    log_progress(f"EB-NeRD small behaviors built: {ebnerd_small_behaviors.shape}")

ebnerd_large_behaviors_raw = pl.concat([
    pl.read_parquet(EBNERD_LARGE / "train" / "behaviors.parquet"),
    pl.read_parquet(EBNERD_LARGE / "validation" / "behaviors.parquet"),
])
ebnerd_large_behaviors = build_ebnerd_behaviors(ebnerd_large_behaviors_raw, "ebnerd_large_")
log_progress(f"EB-NeRD large behaviors built: {ebnerd_large_behaviors.shape}")

if not BUILD_LARGE_ONLY:
    print("ebnerd shape:", ebnerd_behaviors.shape)
    print("ebnerd_small shape:", ebnerd_small_behaviors.shape)
print("ebnerd_large shape:", ebnerd_large_behaviors.shape)
ebnerd_large_behaviors.head(5)

[2026-08-21T06:46:02+00:00] EB-NeRD large behaviors built: (24630275, 7)
ebnerd_large shape: (24630275, 7)


,impression_id,dataset,user_id,impression_time,article_ids_inview,article_ids_clicked,session_id
0,ebnerd_large_47727,ebnerd_large,ebnerd_large_18293,2023-05-21 21:35:07,"[ebnerd_large_9482380, ebnerd_large_9775183, ebnerd_large_9744403, ebnerd_la...",[ebnerd_large_9775183],ebnerd_large_265
1,ebnerd_large_47731,ebnerd_large,ebnerd_large_18293,2023-05-21 21:32:33,"[ebnerd_large_9774557, ebnerd_large_9774516, ebnerd_large_9775331, ebnerd_la...",[ebnerd_large_9759966],ebnerd_large_265
2,ebnerd_large_47736,ebnerd_large,ebnerd_large_18293,2023-05-21 21:33:32,"[ebnerd_large_9759966, ebnerd_large_9774557, ebnerd_large_9775352, ebnerd_la...",[ebnerd_large_9774652],ebnerd_large_265
3,ebnerd_large_47737,ebnerd_large,ebnerd_large_18293,2023-05-21 21:38:17,"[ebnerd_large_9774580, ebnerd_large_9775131, ebnerd_large_9775202, ebnerd_la...",[ebnerd_large_9775184],ebnerd_large_265
4,ebnerd_large_47740,ebnerd_large,ebnerd_large_18293,2023-05-21 21:36:02,"[ebnerd_large_9774826, ebnerd_large_9775171, ebnerd_large_9775076, ebnerd_la...",[ebnerd_large_9774648],ebnerd_large_265


In [5]:
def test_ebnerd_behaviors_unified_schema():
    expected_cols = {
        "impression_id", "dataset", "user_id", "impression_time",
        "article_ids_inview", "article_ids_clicked", "session_id",
    }
    cases = [(ebnerd_large_behaviors, ebnerd_large_behaviors_raw, "ebnerd_large")]
    if not BUILD_LARGE_ONLY:
        cases = [
            (ebnerd_behaviors, ebnerd_behaviors_raw, "ebnerd"),
            (ebnerd_small_behaviors, ebnerd_small_behaviors_raw, "ebnerd_small"),
        ] + cases
    for behaviors, raw, dataset_name in cases:
        assert expected_cols == set(behaviors.columns)
        assert len(behaviors) == len(raw)
        assert behaviors["impression_id"].is_unique
        assert (behaviors["dataset"] == dataset_name).all()
        # zip(), not .apply(axis=1) -- per-row apply's Series-construction
        # overhead makes it painfully slow at ebnerd_large's ~24.6M-row scale
        n_bad = sum(
            1 for clicked, inview in zip(behaviors["article_ids_clicked"], behaviors["article_ids_inview"])
            if not set(clicked).issubset(inview)
        )
        assert n_bad == 0, f"{dataset_name}: found clicked articles absent from the inview candidate set"


test_ebnerd_behaviors_unified_schema()

# free the raw polars frames -- not referenced anywhere past this point.
# ebnerd_large_behaviors_raw specifically is the single largest object built
# in this pipeline (24.6M rows, un-selected columns from the raw parquet
# still attached), so this is the highest-value free in the whole notebook.
if BUILD_LARGE_ONLY:
    del ebnerd_large_behaviors_raw
else:
    del ebnerd_behaviors_raw, ebnerd_small_behaviors_raw, ebnerd_large_behaviors_raw
gc.collect()

print("ok: EB-NeRD unified behaviors schema checks passed")

ok: EB-NeRD unified behaviors schema checks passed


## EB-NeRD -> unified `history`

One row per user (deduplicated across the `train/`/`validation/` history
files, which describe the same fixed pre-collection-window clicks).

In [6]:
def build_ebnerd_history(raw: pl.DataFrame, prefix: str) -> pd.DataFrame:
    out = raw.select(
        (pl.lit(prefix) + pl.col("user_id").cast(pl.Utf8)).alias("user_id"),
        pl.lit(prefix.rstrip("_")).alias("dataset"),
        pl.col("article_id_fixed").list.eval(pl.lit(prefix) + pl.element().cast(pl.Utf8)).alias("article_id_sequence"),
        pl.col("impression_time_fixed").alias("timestamp_sequence"),
        pl.col("read_time_fixed").alias("read_time_sequence"),
        pl.col("scroll_percentage_fixed").alias("scroll_percentage_sequence"),
    )
    return out.to_pandas()


if not BUILD_LARGE_ONLY:
    ebnerd_history_raw = pl.concat([
        pl.read_parquet(EBNERD_DEMO / "train" / "history.parquet"),
        pl.read_parquet(EBNERD_DEMO / "validation" / "history.parquet"),
    ]).unique(subset=["user_id"], keep="first", maintain_order=True)
    ebnerd_history = build_ebnerd_history(ebnerd_history_raw, "ebnerd_")
    log_progress(f"EB-NeRD demo history built: {ebnerd_history.shape}")

    ebnerd_small_history_raw = pl.concat([
        pl.read_parquet(EBNERD_SMALL / "train" / "history.parquet"),
        pl.read_parquet(EBNERD_SMALL / "validation" / "history.parquet"),
    ]).unique(subset=["user_id"], keep="first", maintain_order=True)
    ebnerd_small_history = build_ebnerd_history(ebnerd_small_history_raw, "ebnerd_small_")
    log_progress(f"EB-NeRD small history built: {ebnerd_small_history.shape}")

ebnerd_large_history_raw = pl.concat([
    pl.read_parquet(EBNERD_LARGE / "train" / "history.parquet"),
    pl.read_parquet(EBNERD_LARGE / "validation" / "history.parquet"),
]).unique(subset=["user_id"], keep="first", maintain_order=True)
ebnerd_large_history = build_ebnerd_history(ebnerd_large_history_raw, "ebnerd_large_")
log_progress(f"EB-NeRD large history built: {ebnerd_large_history.shape}")

# free the raw polars frames -- unlike articles/behaviors, no downstream
# cell (not even this table's own test) references the _raw history frames
if BUILD_LARGE_ONLY:
    del ebnerd_large_history_raw
else:
    del ebnerd_history_raw, ebnerd_small_history_raw, ebnerd_large_history_raw
gc.collect()

if not BUILD_LARGE_ONLY:
    print("ebnerd shape (one row per user):", ebnerd_history.shape)
    print("ebnerd_small shape (one row per user):", ebnerd_small_history.shape)
print("ebnerd_large shape (one row per user):", ebnerd_large_history.shape)
ebnerd_large_history.head(5)

[2026-08-21T06:47:03+00:00] EB-NeRD large history built: (974791, 6)
ebnerd_large shape (one row per user): (974791, 6)


,user_id,dataset,article_id_sequence,timestamp_sequence,read_time_sequence,scroll_percentage_sequence
0,ebnerd_large_10029,ebnerd_large,"[ebnerd_large_9735579, ebnerd_large_9739888, ebnerd_large_9739471, ebnerd_la...","[2023-04-28T06:16:57.000000, 2023-04-28T06:17:31.000000, 2023-04-28T06:18:12...","[28.0, 24.0, 11.0, 107.0, 8.0, 7.0, 20.0, 5.0, 4.0, 49.0, 19.0, 10.0, 8.0, 1...","[23.0, 69.0, 27.0, nan, 47.0, 38.0, 100.0, 12.0, 13.0, 84.0, 90.0, 9.0, 22.0..."
1,ebnerd_large_10033,ebnerd_large,"[ebnerd_large_9738139, ebnerd_large_9738263, ebnerd_large_9738139, ebnerd_la...","[2023-04-27T11:11:32.000000, 2023-04-27T11:12:56.000000, 2023-04-27T11:13:00...","[2.0, 2.0, 718.0, 18.0, 26.0, 78.0, 3.0, 11.0, 63.0, 10.0, 5.0, 109.0, 26.0,...","[33.0, 41.0, 33.0, 100.0, 68.0, 38.0, 1.0, 58.0, 100.0, 46.0, 34.0, 96.0, 66..."
2,ebnerd_large_10034,ebnerd_large,"[ebnerd_large_9742693, ebnerd_large_9742686, ebnerd_large_9744016, ebnerd_la...","[2023-04-30T09:46:57.000000, 2023-04-30T09:47:33.000000, 2023-05-01T09:01:54...","[21.0, 103.0, 28.0, 0.0, 5.0, 34.0, 14.0, 14.0, 50.0, 23.0, 4.0, 3.0, 6.0, 2...","[nan, 88.0, 27.0, nan, 23.0, 100.0, 100.0, 22.0, 100.0, 100.0, 32.0, 100.0, ..."
3,ebnerd_large_10041,ebnerd_large,"[ebnerd_large_9739035, ebnerd_large_9738303, ebnerd_large_9737243, ebnerd_la...","[2023-04-27T15:15:28.000000, 2023-04-27T15:16:30.000000, 2023-04-28T04:57:34...","[12.0, 11.0, 3.0, 3.0, 4.0, 13.0, 29.0, 24.0, 6.0, 5.0, 9.0, 20.0, 17.0, 141...","[78.0, 41.0, 4.0, 16.0, 22.0, 32.0, 11.0, 94.0, 87.0, 100.0, 79.0, 76.0, 65...."
4,ebnerd_large_10103,ebnerd_large,"[ebnerd_large_9739035, ebnerd_large_9739164, ebnerd_large_9741803, ebnerd_la...","[2023-04-27T15:37:35.000000, 2023-04-27T15:38:37.000000, 2023-04-29T11:47:41...","[45.0, 8.0, 61.0, 72.0, 56.0, 3.0, 22.0, 16.0, 21.0, 26.0, 57.0, 9.0, 35.0, ...","[100.0, nan, 100.0, 100.0, 100.0, 28.0, 82.0, 100.0, 77.0, 100.0, 71.0, 22.0..."


In [7]:
def test_ebnerd_history_unified_schema():
    expected_cols = {
        "user_id", "dataset", "article_id_sequence",
        "timestamp_sequence", "read_time_sequence", "scroll_percentage_sequence",
    }
    cases = [(ebnerd_large_history, "ebnerd_large")]
    if not BUILD_LARGE_ONLY:
        cases = [(ebnerd_history, "ebnerd"), (ebnerd_small_history, "ebnerd_small")] + cases
    for history, dataset_name in cases:
        assert expected_cols == set(history.columns)
        assert history["user_id"].is_unique
        assert (history["dataset"] == dataset_name).all()
        row = history.iloc[0]
        lengths = {len(row[c]) for c in [
            "article_id_sequence", "timestamp_sequence", "read_time_sequence", "scroll_percentage_sequence",
        ]}
        assert len(lengths) == 1, f"{dataset_name}: the four parallel history arrays must have equal length per user"


test_ebnerd_history_unified_schema()
print("ok: EB-NeRD unified history schema checks passed")

ok: EB-NeRD unified history schema checks passed


## Temporal split (train / val / test) -- EB-NeRD

EB-NeRD ships only two provider splits (train, then validation) -- there's
no provider test set. We treat the provider's validation split as our
held-out **test** set, and carve **val** from the last day of the
provider's train split by time. Cutoffs are real dates read off the raw
files (see `SPEC.md` section 3), not arbitrary constants.

EB-NeRD and MIND are processed as two fully separate families from here on
(EB-NeRD: split -> leakage-check -> write -> free, completely, before
MIND's build even starts) rather than one shared step across all five
datasets -- see the write cell further down for why (memory, at
`ebnerd_large`/`mind_large` scale).

In [8]:
EBNERD_TRAIN_END = pd.Timestamp("2023-05-24 07:00:00")   # last 24h of provider train -> val
EBNERD_TEST_START = pd.Timestamp("2023-05-25 07:00:00")  # == provider validation start


def assign_split(times: pd.Series, train_end: pd.Timestamp, test_start: pd.Timestamp) -> pd.Series:
    return pd.Series(
        np.select([times < train_end, times < test_start], ["train", "val"], default="test"),
        index=times.index,
    )


# EB-NeRD small and EB-NeRD large both share demo's exact provider date range
# (2023-05-18 -> 2023-06-01), so the same cutoffs apply -- verified directly on
# the raw files, not assumed.
if not BUILD_LARGE_ONLY:
    ebnerd_behaviors["split"] = assign_split(ebnerd_behaviors["impression_time"], EBNERD_TRAIN_END, EBNERD_TEST_START)
    ebnerd_small_behaviors["split"] = assign_split(ebnerd_small_behaviors["impression_time"], EBNERD_TRAIN_END, EBNERD_TEST_START)
ebnerd_large_behaviors["split"] = assign_split(ebnerd_large_behaviors["impression_time"], EBNERD_TRAIN_END, EBNERD_TEST_START)
log_progress("temporal split assigned for EB-NeRD large" if BUILD_LARGE_ONLY else "temporal split assigned for EB-NeRD (demo + small + large)")

results = [ebnerd_large_behaviors["split"].value_counts()]
if not BUILD_LARGE_ONLY:
    results = [ebnerd_behaviors["split"].value_counts(), ebnerd_small_behaviors["split"].value_counts()] + results
tuple(results)

[2026-08-21T06:47:07+00:00] temporal split assigned for EB-NeRD large


(split
 test     12566385
 train    10384901
 val       1678989
 Name: count, dtype: int64,)

In [9]:
def test_ebnerd_temporal_split_boundaries():
    cases = [(ebnerd_large_behaviors, "ebnerd_large")]
    if not BUILD_LARGE_ONLY:
        cases = [(ebnerd_behaviors, "ebnerd"), (ebnerd_small_behaviors, "ebnerd_small")] + cases
    for df, name in cases:
        assert set(df["split"]) == {"train", "val", "test"}, f"{name}: missing a split"
        bounds = df.groupby("split")["impression_time"].agg(["min", "max"])
        assert bounds.loc["train", "max"] < bounds.loc["val", "min"], f"{name}: train/val overlap"
        assert bounds.loc["val", "max"] < bounds.loc["test", "min"], f"{name}: val/test overlap"


test_ebnerd_temporal_split_boundaries()
print("ok: EB-NeRD temporal split boundaries are non-overlapping and monotonic (train < val < test)")

ok: EB-NeRD temporal split boundaries are non-overlapping and monotonic (train < val < test)


## No future-click leakage (Q9) -- EB-NeRD

Assert every historical click for a user, in `history.timestamp_sequence`,
happened strictly before that user's earliest logged impression in
`behaviors` (verified on real demo-scale data: 0/1590 violations). MIND's
leakage check is structural instead (no per-click timestamps) -- see the
MIND-specific version of this check later in the notebook.

In [10]:
def compute_ebnerd_leakage_check(history: pd.DataFrame, behaviors: pd.DataFrame) -> pd.DataFrame:
    hist_max = history.assign(
        max_ts=history["timestamp_sequence"].apply(lambda ts: max(ts) if len(ts) else pd.NaT)
    ).set_index("user_id")["max_ts"]
    beh_min = behaviors.groupby("user_id")["impression_time"].min()
    return pd.concat(
        [hist_max.rename("history_max_ts"), beh_min.rename("first_impression")], axis=1
    ).dropna()


if not BUILD_LARGE_ONLY:
    ebnerd_leakage_check = compute_ebnerd_leakage_check(ebnerd_history, ebnerd_behaviors)
    ebnerd_small_leakage_check = compute_ebnerd_leakage_check(ebnerd_small_history, ebnerd_small_behaviors)
ebnerd_large_leakage_check = compute_ebnerd_leakage_check(ebnerd_large_history, ebnerd_large_behaviors)
log_progress("leakage-check table computed for EB-NeRD large" if BUILD_LARGE_ONLY else "leakage-check tables computed for EB-NeRD (demo + small + large)")

ebnerd_large_leakage_check.head()

[2026-08-21T06:49:22+00:00] leakage-check table computed for EB-NeRD large


,history_max_ts,first_impression
user_id,,
ebnerd_large_10029,2023-05-18 06:59:50,2023-05-18 07:00:38
ebnerd_large_10033,2023-05-17 20:22:42,2023-05-18 07:22:27
ebnerd_large_10034,2023-05-16 08:40:52,2023-05-21 13:13:44
ebnerd_large_10041,2023-05-17 14:54:05,2023-05-18 16:00:21
ebnerd_large_10103,2023-05-18 04:52:09,2023-05-20 10:23:25


In [11]:
def test_ebnerd_no_future_click_leakage():
    cases = [(ebnerd_large_leakage_check, "EB-NeRD large")]
    if not BUILD_LARGE_ONLY:
        cases = [(ebnerd_leakage_check, "EB-NeRD"), (ebnerd_small_leakage_check, "EB-NeRD small")] + cases
    for check, name in cases:
        violations = check[check["history_max_ts"] >= check["first_impression"]]
        assert len(violations) == 0, f"{name}: {len(violations)} users have history clicks at/after a logged impression"


test_ebnerd_no_future_click_leakage()

# free -- Q9's test above already validated these, nothing downstream needs them
if BUILD_LARGE_ONLY:
    del ebnerd_large_leakage_check
else:
    del ebnerd_leakage_check, ebnerd_small_leakage_check, ebnerd_large_leakage_check
gc.collect()

print("ok: no future-click leakage detected for EB-NeRD (Q9 behaviour-window boundary)")

ok: no future-click leakage detected for EB-NeRD (Q9 behaviour-window boundary)


## Write the feature store -- EB-NeRD

One directory per dataset under `data/processed/` (gitignored - see Q8),
each with three parquet files plus a manifest recording split cutoffs, row
counts, and a build timestamp for reproducibility.

Written and freed **one dataset at a time**, and as a whole family **before
MIND is even built** -- see the notebook's intro markdown for why (memory,
at `ebnerd_large`/`mind_large` scale). That alone wasn't enough, though:
writing `ebnerd_large` still OOM-killed the kernel even with MIND not yet
built at all, isolating the real cause to `ebnerd_large` on its own --
`pl.from_pandas(behaviors)` on its ~24.6M-row `behaviors` table has to hold
the existing pandas representation *and* build a whole new polars one
simultaneously, and that alone exceeds this machine's 16GB regardless of
what else is loaded. Fixed by writing large tables in row-chunks instead of
one `pl.from_pandas()` call over the whole table: convert+write each chunk
to its own temp parquet file (bounding the transient pandas+polars overlap
to one chunk's size, not the whole table), then merge the chunk files into
the final path via `polars.scan_parquet(...).sink_parquet(...)` -- polars'
streaming engine, which merges without materializing all chunks in memory
at once. Verified directly on synthetic data with the same nested
`list<str>`/`list<datetime>` column shapes that a chunk-merged file reads
back via `pd.read_parquet` with no error -- doesn't reintroduce the earlier
`ArrowNotImplementedError` chunked-array bug, since each chunk file is
itself a single, internally-consistent polars write (exactly the pattern
already proven safe), not multiple row groups written incrementally into
one file.

In [12]:
def write_parquet_chunked(df: pd.DataFrame, path: Path, chunk_rows: int = 300_000) -> None:
    if len(df) <= chunk_rows:
        pl.from_pandas(df).write_parquet(path)
        return

    tmp_dir = path.parent / f"_{path.stem}_chunks_tmp"
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)
    n_chunks = (len(df) + chunk_rows - 1) // chunk_rows
    try:
        for i, start in enumerate(range(0, len(df), chunk_rows)):
            chunk_pl = pl.from_pandas(df.iloc[start:start + chunk_rows])
            chunk_pl.write_parquet(tmp_dir / f"part_{i:04d}.parquet")
            del chunk_pl
            gc.collect()
            log_progress(f"  {path.name}: chunk {i + 1}/{n_chunks} written")
        pl.scan_parquet(tmp_dir / "*.parquet").sink_parquet(path)
    finally:
        shutil.rmtree(tmp_dir)


def write_feature_store(name: str, articles: pd.DataFrame, behaviors: pd.DataFrame, history: pd.DataFrame,
                         train_end: pd.Timestamp, test_start: pd.Timestamp) -> Path:
    out_dir = DATA_OUT / name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Write via polars, not pandas.to_parquet -- pandas/pyarrow's own writer
    # hit a real bug at scale: ebnerd_large's history.parquet (974,791 rows,
    # nested list<datetime>/list<float> columns) wrote "successfully" but then
    # failed on read-back with `ArrowNotImplementedError: Nested data
    # conversions not implemented for chunked array outputs` -- a pyarrow
    # limitation combining chunked nested-list arrays that only surfaces at
    # this scale (ebnerd/ebnerd_small's much smaller history tables never hit
    # it). Verified directly: the same DataFrame written via
    # `pl.from_pandas(...).write_parquet(...)` instead reads back via
    # `pd.read_parquet` with no error, identical data.
    #
    # write_parquet_chunked (not a plain pl.from_pandas(...).write_parquet(...))
    # for the same reason described in this cell's markdown -- see there.
    # chunk_rows dropped from 2,000,000 to 300,000 after a 2M-row chunk's
    # pandas+polars double-representation (~2.6GB) still failed to allocate
    # even with ~12GB nominally free -- Windows/CPython memory fragmentation
    # from the churn of building five datasets' worth of nested-list pandas
    # objects, not raw insufficiency, so a much smaller chunk (needing a much
    # smaller *contiguous* allocation) is the safer target, not a bigger one.
    write_parquet_chunked(articles, out_dir / "articles.parquet")
    write_parquet_chunked(behaviors, out_dir / "behaviors.parquet")
    write_parquet_chunked(history, out_dir / "history.parquet")

    manifest = {
        "dataset": name,
        "schema_version": 1,
        "build_timestamp": datetime.now(timezone.utc).isoformat(),
        "split_cutoffs": {"train_end": str(train_end), "test_start": str(test_start)},
        "row_counts": {
            "articles": len(articles),
            "behaviors": len(behaviors),
            "history": len(history),
            "behaviors_by_split": behaviors["split"].value_counts().to_dict(),
        },
    }
    (out_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    log_progress(f"wrote {name}: {out_dir} (manifest.json now present = fully persisted)")
    return out_dir


# BUILD_LARGE_ONLY: write_feature_store is simply never called for ebnerd/
# ebnerd_small -- their data/processed/ directories are left completely
# untouched, still holding whatever an earlier full run wrote there.
# ebnerd_out_dir/ebnerd_small_out_dir are still set (to the same path
# write_feature_store would have used) purely so the final round-trip test
# below can still verify all five datasets' persisted state, not just the
# ones (re)built this run.
if not BUILD_LARGE_ONLY:
    ebnerd_out_dir = write_feature_store(
        "ebnerd", ebnerd_articles, ebnerd_behaviors, ebnerd_history, EBNERD_TRAIN_END, EBNERD_TEST_START,
    )
    del ebnerd_articles, ebnerd_behaviors, ebnerd_history
    gc.collect()

    ebnerd_small_out_dir = write_feature_store(
        "ebnerd_small", ebnerd_small_articles, ebnerd_small_behaviors, ebnerd_small_history,
        EBNERD_TRAIN_END, EBNERD_TEST_START,
    )
    del ebnerd_small_articles, ebnerd_small_behaviors, ebnerd_small_history
    gc.collect()
else:
    ebnerd_out_dir = DATA_OUT / "ebnerd"              # untouched this run
    ebnerd_small_out_dir = DATA_OUT / "ebnerd_small"  # untouched this run
    log_progress("skipped writing ebnerd/ebnerd_small this run (BUILD_LARGE_ONLY) -- existing outputs left untouched")

ebnerd_large_out_dir = write_feature_store(
    "ebnerd_large", ebnerd_large_articles, ebnerd_large_behaviors, ebnerd_large_history,
    EBNERD_TRAIN_END, EBNERD_TEST_START,
)
del ebnerd_large_articles, ebnerd_large_behaviors, ebnerd_large_history
gc.collect()

print("ebnerd:", ebnerd_out_dir, "(untouched this run)" if BUILD_LARGE_ONLY else "(wrote)")
print("ebnerd_small:", ebnerd_small_out_dir, "(untouched this run)" if BUILD_LARGE_ONLY else "(wrote)")
print("wrote:", ebnerd_large_out_dir)

[2026-08-21T06:49:22+00:00] skipped writing ebnerd/ebnerd_small this run (BUILD_LARGE_ONLY) -- existing outputs left untouched


[2026-08-21T06:49:25+00:00]   behaviors.parquet: chunk 1/83 written


[2026-08-21T06:49:26+00:00]   behaviors.parquet: chunk 2/83 written


[2026-08-21T06:49:27+00:00]   behaviors.parquet: chunk 3/83 written


[2026-08-21T06:49:27+00:00]   behaviors.parquet: chunk 4/83 written


...

[2026-08-21T06:51:11+00:00]   history.parquet: chunk 2/4 written


[2026-08-21T06:51:14+00:00]   history.parquet: chunk 3/4 written


[2026-08-21T06:51:15+00:00]   history.parquet: chunk 4/4 written


[2026-08-21T06:51:27+00:00] wrote ebnerd_large: C:\Users\HP\cs4406m26-assignment1c1\data\processed\ebnerd_large (manifest.json now present = fully persisted)


ebnerd: C:\Users\HP\cs4406m26-assignment1c1\data\processed\ebnerd (untouched this run)
ebnerd_small: C:\Users\HP\cs4406m26-assignment1c1\data\processed\ebnerd_small (untouched this run)
wrote: C:\Users\HP\cs4406m26-assignment1c1\data\processed\ebnerd_large


## MIND -> unified `articles`

`body` is always null - MSN's licensing terms mean the full article text was
never distributed (see `README.md`); it's a genuine dataset limitation, not a
parsing gap.

Read via `polars.read_csv(..., quote_char=None)`, not `pandas.read_csv` --
this is a genuine correctness fix, not just speed. `news.tsv` is a raw TSV
with no CSV-style escaping convention, but pandas' C parser defaults to
CSV-quote semantics (`quotechar='"'`) regardless, which silently strips
literal `"` characters from ~62 titles / ~628 abstracts across MIND's
catalog (e.g. a title like `"It changed everything," Oklahoma woman...`
loses both quote marks). `quote_char=None` disables that interpretation, so
these fields round-trip byte-for-byte. Verified against the already-built
`mind`/`mind_large` catalogs directly. This has no effect on BM25 (its
`\w+` tokenizer already discards punctuation, so tokens are identical either
way) and a negligible one on embeddings (<1% of articles, punctuation-only
difference, averaged over tens of thousands of impressions in any reported
metric) -- not something that requires re-running already-computed Q2-Q5
results, just a small, worth-noting correctness improvement.

In [13]:
MIND_NEWS_COLS = [
    "news_id", "category", "subcategory", "title", "abstract",
    "url", "title_entities", "abstract_entities",
]


def load_mind_news_raw(train_dir: Path, dev_dir: Path) -> pl.DataFrame:
    schema = {c: pl.Utf8 for c in MIND_NEWS_COLS}
    return pl.concat([
        pl.read_csv(train_dir / "news.tsv", separator="\t", has_header=False,
                     new_columns=MIND_NEWS_COLS, schema_overrides=schema, quote_char=None),
        pl.read_csv(dev_dir / "news.tsv", separator="\t", has_header=False,
                     new_columns=MIND_NEWS_COLS, schema_overrides=schema, quote_char=None),
    ]).unique(subset=["news_id"], keep="first", maintain_order=True)


def build_mind_articles(raw: pl.DataFrame, prefix: str) -> pd.DataFrame:
    out = raw.select(
        (pl.lit(prefix) + pl.col("news_id")).alias("article_id"),
        pl.lit(prefix.rstrip("_")).alias("dataset"),
        pl.col("title"),
        pl.col("abstract"),
        pl.lit(None, dtype=pl.Utf8).alias("body"),
        pl.col("category"),
        pl.col("subcategory"),
        pl.lit(None, dtype=pl.Datetime).alias("published_time"),
    )
    return out.to_pandas()


if not BUILD_LARGE_ONLY:
    mind_articles_raw = load_mind_news_raw(MIND_TRAIN, MIND_DEV)
    mind_articles = build_mind_articles(mind_articles_raw, "mind_")
    log_progress(f"MIND articles built: {mind_articles.shape}")

mind_large_articles_raw = load_mind_news_raw(MIND_LARGE_TRAIN, MIND_LARGE_DEV)
mind_large_articles = build_mind_articles(mind_large_articles_raw, "mind_large_")
log_progress(f"MIND large articles built: {mind_large_articles.shape}")

# free the raw polars frames -- not referenced anywhere past this point
if BUILD_LARGE_ONLY:
    del mind_large_articles_raw
else:
    del mind_articles_raw, mind_large_articles_raw
gc.collect()

if not BUILD_LARGE_ONLY:
    print("mind shape:", mind_articles.shape)
print("mind_large shape:", mind_large_articles.shape)
mind_large_articles.head(5)

[2026-08-21T06:51:38+00:00] MIND large articles built: (104151, 8)
mind_large shape: (104151, 8)


,article_id,dataset,title,abstract,body,category,subcategory,published_time
0,mind_large_N88753,mind_large,"The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By","Shop the notebooks, jackets, and more that the royals can't live without.",NaN,lifestyle,lifestyleroyals,NaT
1,mind_large_N45436,mind_large,Walmart Slashes Prices on Last-Generation iPads,Apple's new iPad releases bring big deals on last year's models.,NaN,news,newsscienceandtechnology,NaT
2,mind_large_N23144,mind_large,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding you back and keeping you from sh...,NaN,health,weightloss,NaT
3,mind_large_N86255,mind_large,Dispose of unwanted prescription drugs during the DEA's Take Back Day,NaN,NaN,health,medical,NaT
4,mind_large_N93187,mind_large,The Cost of Trump's Aid Freeze in the Trenches of Ukraine's War,Lt. Ivan Molchanets peeked over a parapet of sand bags at the front line of ...,NaN,news,newsworld,NaT


In [14]:
def test_mind_articles_unified_schema():
    expected_cols = {
        "article_id", "dataset", "title", "abstract", "body",
        "category", "subcategory", "published_time",
    }
    cases = [(mind_large_articles, "mind_large")]
    if not BUILD_LARGE_ONLY:
        cases = [(mind_articles, "mind")] + cases
    for articles, dataset_name in cases:
        assert expected_cols == set(articles.columns)
        assert articles["article_id"].is_unique
        assert (articles["dataset"] == dataset_name).all()
        assert articles["body"].isna().all(), "MIND must never have body text (licensing)"


test_mind_articles_unified_schema()
print("ok: MIND unified articles schema checks passed")

ok: MIND unified articles schema checks passed


## MIND -> unified `behaviors`

`impressions` packs candidates and labels into one string (`news_id-label`
tokens); we split it into the same `article_ids_inview` / `article_ids_clicked`
shape as EB-NeRD.

Parsed via an explode -> transform -> group-by -> join-back pattern instead
of a per-row Python function (the original `split_impressions` +
`.map()`) -- at `mind_large`'s ~2.6M-row scale, exploding each impression's
tokens into their own rows lets every step (the id/label split, the
click-label filter, the re-aggregation into per-row lists) run as a
vectorized polars operation instead of a Python loop per row per token.
`id`/`label` are recovered via `str.slice` from each end of the token
(`"N55689-1"` -> id `"N55689"`, label `"1"`) rather than a plain split on
`"-"`, so it stays correct even if an id ever contained a literal `-`
(matches the original `rsplit("-", 1)` semantics exactly). Explicitly
`.sort("row_id")` after the joins to guarantee the output keeps the raw
file's original row order -- polars left-joins aren't documented to
guarantee this, and Q5's submission format depends on it.

In [15]:
MIND_BEHAVIORS_COLS = ["impression_id", "user_id", "time", "history", "impressions"]


def load_mind_behaviors_raw(train_dir: Path, dev_dir: Path) -> pl.DataFrame:
    # provider train/dev files both restart impression_id at 1 -> must namespace by
    # source file too, not just dataset, or concatenating collides train with dev.
    schema = {c: pl.Utf8 for c in MIND_BEHAVIORS_COLS}
    return pl.concat([
        pl.read_csv(train_dir / "behaviors.tsv", separator="\t", has_header=False,
                     new_columns=MIND_BEHAVIORS_COLS, schema_overrides=schema, quote_char=None)
          .with_columns(pl.lit("train").alias("source_split")),
        pl.read_csv(dev_dir / "behaviors.tsv", separator="\t", has_header=False,
                     new_columns=MIND_BEHAVIORS_COLS, schema_overrides=schema, quote_char=None)
          .with_columns(pl.lit("dev").alias("source_split")),
    ])


def build_mind_behaviors(raw: pl.DataFrame, prefix: str) -> pd.DataFrame:
    raw = raw.with_row_index("row_id")

    # one row per (impression, candidate) token, so id/label extraction and
    # the click filter are vectorized instead of a Python loop per impression
    tokens = (
        raw.select("row_id", pl.col("impressions").str.split(" ").alias("token"))
        .explode("token")
        .with_columns(
            pl.col("token").str.slice(0, pl.col("token").str.len_chars() - 2).alias("news_id"),
            pl.col("token").str.slice(-1).cast(pl.Int8).alias("label"),
        )
        .with_columns((pl.lit(prefix) + pl.col("news_id")).alias("article_id"))
    )
    inview = tokens.group_by("row_id").agg(pl.col("article_id").alias("article_ids_inview"))
    clicked = (
        tokens.filter(pl.col("label") == 1)
        .group_by("row_id")
        .agg(pl.col("article_id").alias("article_ids_clicked"))
    )

    out = (
        raw.join(inview, on="row_id", how="left")
        .join(clicked, on="row_id", how="left")
        .sort("row_id")  # left joins aren't guaranteed order-preserving -- re-sort explicitly
        .with_columns(pl.col("article_ids_clicked").fill_null([]))
        .select(
            (pl.lit(prefix) + pl.col("source_split") + "_" + pl.col("impression_id")).alias("impression_id"),
            pl.lit(prefix.rstrip("_")).alias("dataset"),
            (pl.lit(prefix) + pl.col("user_id")).alias("user_id"),
            pl.col("time").str.strptime(pl.Datetime, "%m/%d/%Y %I:%M:%S %p").alias("impression_time"),
            pl.col("article_ids_inview"),
            pl.col("article_ids_clicked"),
            pl.lit(None, dtype=pl.Utf8).alias("session_id"),
        )
    )
    return out.to_pandas()


if not BUILD_LARGE_ONLY:
    mind_behaviors_raw = load_mind_behaviors_raw(MIND_TRAIN, MIND_DEV)
    mind_behaviors = build_mind_behaviors(mind_behaviors_raw, "mind_")
    log_progress(f"MIND behaviors built: {mind_behaviors.shape}")

mind_large_behaviors_raw = load_mind_behaviors_raw(MIND_LARGE_TRAIN, MIND_LARGE_DEV)
mind_large_behaviors = build_mind_behaviors(mind_large_behaviors_raw, "mind_large_")
log_progress(f"MIND large behaviors built: {mind_large_behaviors.shape}")

if not BUILD_LARGE_ONLY:
    print("mind shape:", mind_behaviors.shape)
print("mind_large shape:", mind_large_behaviors.shape)
mind_large_behaviors.head(5)

C:\Users\HP\AppData\Local\Temp\ipykernel_7944\1916523307.py:25: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("token")


[2026-08-21T06:52:05+00:00] MIND large behaviors built: (2609219, 7)
mind_large shape: (2609219, 7)


,impression_id,dataset,user_id,impression_time,article_ids_inview,article_ids_clicked,session_id
0,mind_large_train_1,mind_large,mind_large_U87243,2019-11-10 11:30:54,"[mind_large_N78206, mind_large_N26368, mind_large_N7578, mind_large_N58592, ...","[mind_large_N94157, mind_large_N78699, mind_large_N71090, mind_large_N31174]",NaN
1,mind_large_train_2,mind_large,mind_large_U598644,2019-11-12 13:45:29,"[mind_large_N47996, mind_large_N82719, mind_large_N117066, mind_large_N8491,...","[mind_large_N25587, mind_large_N36266]",NaN
2,mind_large_train_3,mind_large,mind_large_U532401,2019-11-13 11:23:03,"[mind_large_N103852, mind_large_N53474, mind_large_N127836, mind_large_N47925]",[mind_large_N47925],NaN
3,mind_large_train_4,mind_large,mind_large_U593596,2019-11-12 12:24:09,"[mind_large_N38902, mind_large_N76434, mind_large_N71593, mind_large_N100073...",[mind_large_N114935],NaN
4,mind_large_train_5,mind_large,mind_large_U239687,2019-11-14 20:03:01,"[mind_large_N76209, mind_large_N48841, mind_large_N67937, mind_large_N62235,...",[mind_large_N86258],NaN


In [16]:
def test_mind_behaviors_unified_schema():
    expected_cols = {
        "impression_id", "dataset", "user_id", "impression_time",
        "article_ids_inview", "article_ids_clicked", "session_id",
    }
    cases = [(mind_large_behaviors, mind_large_behaviors_raw, "mind_large")]
    if not BUILD_LARGE_ONLY:
        cases = [(mind_behaviors, mind_behaviors_raw, "mind")] + cases
    for behaviors, raw, dataset_name in cases:
        assert expected_cols == set(behaviors.columns)
        assert len(behaviors) == len(raw)
        assert behaviors["impression_id"].is_unique
        assert (behaviors["dataset"] == dataset_name).all()
        assert behaviors["session_id"].isna().all(), "MIND has no session concept"
        # zip(), not .apply(axis=1) -- see the matching EB-NeRD check for why
        n_bad = sum(
            1 for clicked, inview in zip(behaviors["article_ids_clicked"], behaviors["article_ids_inview"])
            if not set(clicked).issubset(inview)
        )
        assert n_bad == 0, f"{dataset_name}: found clicked articles absent from the inview candidate set"


test_mind_behaviors_unified_schema()
print("ok: MIND unified behaviors schema checks passed")

ok: MIND unified behaviors schema checks passed


## MIND -> unified `history`

MIND has no dedicated history file - each `behaviors.tsv` row repeats the
same pre-log-window `history` string for a given user. We collapse that into
one row per user and **fail the build** if any user's `history` string isn't
identical across all their rows, since that would mean history was somehow
computed per-impression instead of being a fixed snapshot (the Q9 leakage
pattern this pipeline must never produce). Same polars group-by-based
consistency check as before, just expressed as a `group_by().agg(n_unique())`
+ `filter()` instead of a pandas `groupby().nunique()` -- avoids materializing
a Python dict keyed by up to ~1M users (`mind_large`'s scale) to build the
per-user sequences.

In [17]:
def build_mind_history(raw: pl.DataFrame, prefix: str) -> pd.DataFrame:
    non_null = raw.drop_nulls(subset=["history"])
    inconsistent = (
        non_null.group_by("user_id").agg(pl.col("history").n_unique().alias("n_unique"))
        .filter(pl.col("n_unique") > 1)
    )
    if len(inconsistent):
        raise ValueError(
            f"{len(inconsistent)} MIND users have a history string that differs "
            "across their own impression rows - history must be a fixed "
            "pre-window snapshot, not something derived per-impression"
        )

    first_history = non_null.group_by("user_id").agg(pl.col("history").first().alias("history"))
    all_users = raw.select("user_id").unique(maintain_order=True)

    out = (
        all_users.join(first_history, on="user_id", how="left")
        .select(
            (pl.lit(prefix) + pl.col("user_id")).alias("user_id"),
            pl.lit(prefix.rstrip("_")).alias("dataset"),
            pl.col("history").str.split(" ").fill_null([])
              .list.eval(pl.lit(prefix) + pl.element()).alias("article_id_sequence"),
            pl.lit(None, dtype=pl.List(pl.Datetime)).alias("timestamp_sequence"),
            pl.lit(None, dtype=pl.List(pl.Float64)).alias("read_time_sequence"),
            pl.lit(None, dtype=pl.List(pl.Float64)).alias("scroll_percentage_sequence"),
        )
    )
    return out.to_pandas()


if not BUILD_LARGE_ONLY:
    mind_history = build_mind_history(mind_behaviors_raw, "mind_")
    log_progress(f"MIND history built: {mind_history.shape}")

mind_large_history = build_mind_history(mind_large_behaviors_raw, "mind_large_")
log_progress(f"MIND large history built: {mind_large_history.shape}")

if not BUILD_LARGE_ONLY:
    print("mind shape (one row per user):", mind_history.shape)
print("mind_large shape (one row per user):", mind_large_history.shape)
mind_large_history.head(5)

[2026-08-21T06:52:11+00:00] MIND large history built: (750434, 6)
mind_large shape (one row per user): (750434, 6)


,user_id,dataset,article_id_sequence,timestamp_sequence,read_time_sequence,scroll_percentage_sequence
0,mind_large_U87243,mind_large,"[mind_large_N8668, mind_large_N39081, mind_large_N65259, mind_large_N79529, ...",None,None,None
1,mind_large_U598644,mind_large,"[mind_large_N56056, mind_large_N8726, mind_large_N70353, mind_large_N67998, ...",None,None,None
2,mind_large_U532401,mind_large,"[mind_large_N128643, mind_large_N87446, mind_large_N122948, mind_large_N9375...",None,None,None
3,mind_large_U593596,mind_large,"[mind_large_N31043, mind_large_N39592, mind_large_N4104, mind_large_N8223, m...",None,None,None
4,mind_large_U239687,mind_large,"[mind_large_N65250, mind_large_N122359, mind_large_N71723, mind_large_N53796...",None,None,None


In [18]:
def test_mind_history_unified_schema():
    expected_cols = {
        "user_id", "dataset", "article_id_sequence",
        "timestamp_sequence", "read_time_sequence", "scroll_percentage_sequence",
    }
    cases = [(mind_large_history, mind_large_behaviors, "mind_large")]
    if not BUILD_LARGE_ONLY:
        cases = [(mind_history, mind_behaviors, "mind")] + cases
    for history, behaviors, dataset_name in cases:
        assert expected_cols == set(history.columns)
        assert history["user_id"].is_unique
        assert (history["dataset"] == dataset_name).all()
        # every user seen in behaviors gets a history row, even if empty (cold-start)
        assert set(behaviors["user_id"]) == set(history["user_id"])
        assert history["timestamp_sequence"].isna().all(), "MIND never provides per-click timestamps"


test_mind_history_unified_schema()
print("ok: MIND unified history schema checks passed")

ok: MIND unified history schema checks passed


## Temporal split (train / val / test) -- MIND

MIND ships only two provider splits (train, then dev) -- there's no
provider test set. We treat the provider's dev split as our held-out
**test** set, and carve **val** from the last day of the provider's train
split by time. Cutoffs are real dates read off the raw files (see
`SPEC.md` section 3), not arbitrary constants. `assign_split` is already
defined (EB-NeRD's temporal-split cell above), reused directly here.

In [19]:
MIND_TRAIN_END = pd.Timestamp("2019-11-14 00:00:00")     # last day of provider train -> val
MIND_TEST_START = pd.Timestamp("2019-11-15 00:00:00")    # == provider dev start

# MINDlarge shares MINDsmall's exact date range (2019-11-09 -> 2019-11-16) --
# verified directly, not assumed.
if not BUILD_LARGE_ONLY:
    mind_behaviors["split"] = assign_split(mind_behaviors["impression_time"], MIND_TRAIN_END, MIND_TEST_START)
mind_large_behaviors["split"] = assign_split(mind_large_behaviors["impression_time"], MIND_TRAIN_END, MIND_TEST_START)
log_progress("temporal split assigned for MIND large" if BUILD_LARGE_ONLY else "temporal split assigned for MIND (small + large)")

results = [mind_large_behaviors["split"].value_counts()]
if not BUILD_LARGE_ONLY:
    results = [mind_behaviors["split"].value_counts()] + results
tuple(results)

[2026-08-21T06:52:14+00:00] temporal split assigned for MIND large


(split
 train    1801231
 val       431517
 test      376471
 Name: count, dtype: int64,)

In [20]:
def test_mind_temporal_split_boundaries():
    cases = [(mind_large_behaviors, "mind_large")]
    if not BUILD_LARGE_ONLY:
        cases = [(mind_behaviors, "mind")] + cases
    for df, name in cases:
        assert set(df["split"]) == {"train", "val", "test"}, f"{name}: missing a split"
        bounds = df.groupby("split")["impression_time"].agg(["min", "max"])
        assert bounds.loc["train", "max"] < bounds.loc["val", "min"], f"{name}: train/val overlap"
        assert bounds.loc["val", "max"] < bounds.loc["test", "min"], f"{name}: val/test overlap"


test_mind_temporal_split_boundaries()
print("ok: MIND temporal split boundaries are non-overlapping and monotonic (train < val < test)")

ok: MIND temporal split boundaries are non-overlapping and monotonic (train < val < test)


## No future-click leakage (Q9) -- MIND

MIND has no history timestamps, so timestamp comparison isn't possible (see
EB-NeRD's version of this check above for that approach). The invariant
that matters here already ran when `mind_history`/`mind_large_history` were
built (a `ValueError` would have aborted the build if any user's history
string differed across impressions) -- asserted again explicitly here, so
the rebuild gate depends on a visible test, not just a constructor side
effect.

In [21]:
def compute_mind_history_string_counts(behaviors_raw: pl.DataFrame) -> pd.Series:
    counts = (
        behaviors_raw.drop_nulls(subset=["history"])
        .group_by("user_id")
        .agg(pl.col("history").n_unique().alias("n_unique"))
    )
    return counts.to_pandas().set_index("user_id")["n_unique"]


if not BUILD_LARGE_ONLY:
    mind_history_string_counts = compute_mind_history_string_counts(mind_behaviors_raw)
mind_large_history_string_counts = compute_mind_history_string_counts(mind_large_behaviors_raw)
log_progress("leakage-check table computed for MIND large" if BUILD_LARGE_ONLY else "leakage-check tables computed for MIND (small + large)")

# free the raw polars frames -- this is their last use anywhere in the
# pipeline (build_mind_history already consumed them earlier)
if BUILD_LARGE_ONLY:
    del mind_large_behaviors_raw
else:
    del mind_behaviors_raw, mind_large_behaviors_raw
gc.collect()

mind_large_history_string_counts.head()

[2026-08-21T06:52:16+00:00] leakage-check table computed for MIND large


user_id
U450585    1
U161469    1
U119317    1
U140077    1
U214648    1
Name: n_unique, dtype: uint32

In [22]:
def test_mind_no_future_click_leakage():
    cases = [(mind_large_history_string_counts, "MIND large")]
    if not BUILD_LARGE_ONLY:
        cases = [(mind_history_string_counts, "MIND")] + cases
    for counts, name in cases:
        inconsistent = counts[counts > 1]
        assert len(inconsistent) == 0, f"{name}: {len(inconsistent)} users have a history string that varies across impressions"


test_mind_no_future_click_leakage()

# free -- Q9's test above already validated these, nothing downstream needs them
if BUILD_LARGE_ONLY:
    del mind_large_history_string_counts
else:
    del mind_history_string_counts, mind_large_history_string_counts
gc.collect()

print("ok: no future-click leakage detected for MIND (Q9 behaviour-window boundary)")

ok: no future-click leakage detected for MIND (Q9 behaviour-window boundary)


## Write the feature store -- MIND

Same `write_feature_store` function already defined above (EB-NeRD's write
cell) -- reused directly, written and freed one dataset at a time here too.

In [23]:
# BUILD_LARGE_ONLY: write_feature_store is simply never called for mind --
# its data/processed/mind/ directory is left completely untouched.
# mind_out_dir is still set so the final round-trip test can still verify
# all five datasets' persisted state.
if not BUILD_LARGE_ONLY:
    mind_out_dir = write_feature_store(
        "mind", mind_articles, mind_behaviors, mind_history, MIND_TRAIN_END, MIND_TEST_START,
    )
    del mind_articles, mind_behaviors, mind_history
    gc.collect()
else:
    mind_out_dir = DATA_OUT / "mind"  # untouched this run
    log_progress("skipped writing mind this run (BUILD_LARGE_ONLY) -- existing outputs left untouched")

mind_large_out_dir = write_feature_store(
    "mind_large", mind_large_articles, mind_large_behaviors, mind_large_history, MIND_TRAIN_END, MIND_TEST_START,
)
del mind_large_articles, mind_large_behaviors, mind_large_history
gc.collect()

print("mind:", mind_out_dir, "(untouched this run)" if BUILD_LARGE_ONLY else "(wrote)")
print("wrote:", mind_large_out_dir)

[2026-08-21T06:52:16+00:00] skipped writing mind this run (BUILD_LARGE_ONLY) -- existing outputs left untouched


[2026-08-21T06:52:17+00:00]   behaviors.parquet: chunk 1/9 written


[2026-08-21T06:52:19+00:00]   behaviors.parquet: chunk 2/9 written


[2026-08-21T06:52:20+00:00]   behaviors.parquet: chunk 3/9 written


[2026-08-21T06:52:21+00:00]   behaviors.parquet: chunk 4/9 written


...

[2026-08-21T06:52:30+00:00]   history.parquet: chunk 1/3 written


[2026-08-21T06:52:30+00:00]   history.parquet: chunk 2/3 written


[2026-08-21T06:52:31+00:00]   history.parquet: chunk 3/3 written


[2026-08-21T06:52:31+00:00] wrote mind_large: C:\Users\HP\cs4406m26-assignment1c1\data\processed\mind_large (manifest.json now present = fully persisted)


mind: C:\Users\HP\cs4406m26-assignment1c1\data\processed\mind (untouched this run)
wrote: C:\Users\HP\cs4406m26-assignment1c1\data\processed\mind_large


In [24]:
def test_feature_store_roundtrip():
    for name, out_dir in [
        ("ebnerd", ebnerd_out_dir),
        ("ebnerd_small", ebnerd_small_out_dir),
        ("ebnerd_large", ebnerd_large_out_dir),
        ("mind", mind_out_dir),
        ("mind_large", mind_large_out_dir),
    ]:
        manifest_path = out_dir / "manifest.json"
        assert manifest_path.exists(), f"{name}: missing manifest.json"
        manifest = json.loads(manifest_path.read_text())
        assert set(manifest["row_counts"]["behaviors_by_split"]) == {"train", "val", "test"}

        # Compare the reloaded parquet files against manifest.json's own
        # recorded row counts, not the original in-memory DataFrames -- those
        # were deliberately freed right after being written (see the write
        # cell above) to keep peak memory bounded at ebnerd_large/mind_large
        # scale. manifest.json was itself written from `len(articles)` etc.
        # at write time, so it's an equally trustworthy source of truth.
        for fname, table_key in [
            ("articles.parquet", "articles"), ("behaviors.parquet", "behaviors"), ("history.parquet", "history"),
        ]:
            path = out_dir / fname
            assert path.exists(), f"{name}: missing {fname}"
            reloaded = pd.read_parquet(path)
            expected_len = manifest["row_counts"][table_key]
            assert len(reloaded) == expected_len, f"{name}: {fname} row count mismatch after roundtrip"


test_feature_store_roundtrip()
log_progress("BUILD COMPLETE -- feature store round-trips correctly for all five dataset tracks")
print("ok: feature store round-trips correctly for all five dataset tracks")

[2026-08-21T06:53:52+00:00] BUILD COMPLETE -- feature store round-trips correctly for all five dataset tracks
ok: feature store round-trips correctly for all five dataset tracks


# Manual Review Complete